In [ ]:
# 3) Qual o ticket médio de pedidos por segmento de cliente (B2C vs B2B)? Existe diferença estatisticamente relevante?

import pandas as pd
import scipy.stats as stats
from pathlib import Path
from IPython.display import display

DATA_DIR = Path('../data')

def brl(valor: float) -> str: 
    return f"R$ {valor:,.2f}".replace(',', 'v').replace('.', ',').replace('v', '.')

def load_clean(tabela: str) -> pd.DataFrame:
    path = DATA_DIR / f'{tabela}_limpo.csv'
    if not path.exists(): raise FileNotFoundError("Erro: Rode o script ETL (Q6) primeiro.")
    return pd.read_csv(path)

df_pedidos  = load_clean('pedidos')
df_clientes = load_clean('clientes')

df_merged = df_pedidos.merge(
    df_clientes[['id', 'segmento']],
    left_on='cliente_id', right_on='id',
    how='inner', validate='many_to_one',
)

ticket_medio = (
    df_merged.groupby('segmento')['valor_total']
    .mean().reset_index()
    .rename(columns={'segmento': 'Segmento', 'valor_total': '_media'})
)
ticket_medio['Ticket Médio'] = ticket_medio['_media'].apply(brl)

print("─── Ticket Médio por Segmento ───")
display(ticket_medio[['Segmento', 'Ticket Médio']].style.hide(axis='index'))

b2c = df_merged[df_merged['segmento'] == 'B2C']['valor_total'].dropna()
b2b = df_merged[df_merged['segmento'] == 'B2B']['valor_total'].dropna()

t_stat, p_valor = stats.ttest_ind(b2c, b2b, equal_var=False)

print("\n─── Resultado do Teste Estatístico (Welch t-test) ───")
print(f"Estatística T : {t_stat:.2f}")
print(f"P-Valor       : {p_valor:.6f}")
print(f"N (B2C)       : {len(b2c):,}  |  N (B2B): {len(b2b):,}")

if p_valor < 0.05:
    print("\nCONCLUSÃO:")
    print("EXISTE diferença estatisticamente relevante (p-valor < 0.05).")
    print("O comportamento de gasto de B2B e B2C é de fato distinto na NovaShop.")
else:
    print("\nCONCLUSÃO:")
    print("NÃO existe diferença estatisticamente relevante (p-valor ≥ 0.05).")
    print("A diferença observada pode ser flutuação aleatória da amostra.")

─── Ticket Médio por Segmento ───


Segmento,Ticket Médio
B2B,"R$ 7.778,34"
B2C,"R$ 1.264,72"



─── Resultado do Teste Estatístico (Welch t-test) ───
Estatística T : -85.62
P-Valor       : 0.000000
N (B2C)       : 11,920  |  N (B2B): 3,001

✅ CONCLUSÃO:
EXISTE diferença estatisticamente relevante (p-valor < 0.05).
O comportamento de gasto de B2B e B2C é de fato distinto na NovaShop.
